In [0]:
-- Position - Scoring by Position
-- Goals, assists and points by position.
-- Per-game rates are per player per game.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,       -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,  -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type       -- 2 = regular season, 3 = playoffs
),
player_games AS (                        -- one row per skater per game dressed (game rosters)
  SELECT r.game_id, r.player_id, p.full_name, r.position_code, r.sweater_number,
         r.team_id, t.abbrev AS team, o.abbrev AS opponent, r.team_id = g.home_team_id AS is_home,
         g.season, g.game_date, g.home_team_id,
         CASE WHEN IF(r.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game   g ON r.game_id   = g.game_id
  JOIN nhl.gold.dim_player p ON r.player_id = p.player_id
  JOIN nhl.gold.dim_team   t ON r.team_id   = t.team_id
  JOIN nhl.gold.dim_team   o ON o.team_id   = IF(r.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
    AND r.position_code <> 'G'
),
events AS (                              -- plays where one of those players had a role
  SELECT pg.player_id AS pid,
         IF(f.event_owner_team_id = pg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength,
         f.*
  FROM nhl.gold.fact_event f
  JOIN player_games pg
    ON f.game_id = pg.game_id
   AND pg.player_id IN (f.scoring_player_id, f.assist1_player_id, f.assist2_player_id,
                        f.shooting_player_id, f.blocking_player_id, f.hitting_player_id,
                        f.hittee_player_id, f.winning_player_id, f.losing_player_id,
                        f.committed_by_player_id, f.drawn_by_player_id, f.player_id)
  WHERE COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
),
stats AS (
  SELECT CASE pg.position_code WHEN 'C' THEN 'Center' WHEN 'L' THEN 'Left wing'
              WHEN 'R' THEN 'Right wing' WHEN 'D' THEN 'Defense' END AS position_group,
         COUNT(DISTINCT pg.player_id) AS players,
         COUNT(DISTINCT pg.game_id, pg.player_id) AS games_played,
         COUNT_IF(e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid) AS goals,
         COUNT_IF(e.type_desc_key = 'goal' AND e.pid IN (e.assist1_player_id, e.assist2_player_id)) AS assists,
         COUNT_IF(e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid AND e.strength = 'EV') AS ev_goals,
         COUNT_IF(e.type_desc_key = 'goal' AND e.strength = 'EV'
               AND e.pid IN (e.assist1_player_id, e.assist2_player_id)) AS ev_assists,
         COUNT_IF(e.type_desc_key = 'goal' AND e.scoring_player_id = e.pid AND e.strength = 'PP') AS pp_goals,
         COUNT_IF(e.type_desc_key = 'goal' AND e.strength = 'PP'
               AND e.pid IN (e.assist1_player_id, e.assist2_player_id)) AS pp_assists
  FROM player_games pg
  LEFT JOIN events e ON e.pid = pg.player_id AND e.game_id = pg.game_id
  GROUP BY CASE pg.position_code WHEN 'C' THEN 'Center' WHEN 'L' THEN 'Left wing'
              WHEN 'R' THEN 'Right wing' WHEN 'D' THEN 'Defense' END
),
final AS (
  SELECT position_group,
         players,
         games_played,
         goals,
         assists,
         goals + assists AS points,
         ROUND(goals / games_played, 2) AS goals_per_game,
         ROUND(assists / games_played, 2) AS assists_per_game,
         ROUND((goals + assists) / games_played, 2) AS points_per_game,
         ev_goals + ev_assists AS ev_points,
         pp_goals + pp_assists AS pp_points
  FROM stats
)
SELECT *
FROM final
ORDER BY position_group;